# Notebook 23 — contact sheets for the `strict_v1`-CORRECT spot check

The 2026-08-11 human audit read all 104 of Qwen's `genuinely_wrong` items and
found ~74% of them to be the scoring pipeline rather than the model. That
correction is **one-sided**: only items the frozen rule called *wrong* were
looked at, so only false *negatives* could be found. A reviewer's first
question is the obvious one —

> *You only looked for false negatives. Did you look for false positives?*

This notebook builds the sheets that answer it. It renders the 40 randomly
drawn items from `reference/audit/spotcheck_40_qwen_strict_v1_correct_20260811.csv`
— items `strict_v1` scored **CORRECT** — so they can be read for false passes.

**4 of the 40 are already-known `false_pass_removed` items** (31, 117, 239,
294), marked in the captions. They are the calibration check: if the human
pass catches them, the method works; if it misses them, the resulting rate
should not be trusted.

Selection was random with a fixed seed (`20260811`), never by hand, so what
comes out is an estimate rather than a collection of interesting cases.

No GPU. Reads the dataset and one results CSV; runs no generation.

In [1]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and an existing results CSV, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.plotting
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy. Fix before trusting output.")
print("SymPy LaTeX parser OK")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.4 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot
SymPy LaTeX parser OK


In [2]:
import pandas as pd

RUN_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
SHEETS = {
    # (csv in the cloned repo, Drive subfolder, page stem)
    "first40": ("repo/reference/audit/spotcheck_40_qwen_strict_v1_correct_20260811.csv",
                "spotcheck_strict_v1_correct", "spotcheck_correct"),
    "extra60": ("repo/reference/audit/spotcheck_extra60_qwen_strict_v1_correct_20260812.csv",
                "spotcheck_strict_v1_correct_extra60", "spotcheck_correct_extra"),
}
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5

run = pd.read_csv(f"{RESULTS_DIR}/{RUN_CSV}")
sheets = {k: (pd.read_csv(v[0]), v[1], v[2]) for k, v in SHEETS.items()}
print(f"run  {len(run)} rows, model={run['model_id'].unique().tolist()}")
for k, (df, sub, _) in sheets.items():
    print(f"  {k:8s} {len(df):3d} items, "
          f"{int(df['known_false_pass'].sum())} known false passes -> {sub}")

# The two draws must not overlap, or the combined false-pass rate double
# counts and stops being an unbiased estimate of the 141.
a, b = sheets["first40"][0]["item"], sheets["extra60"][0]["item"]
assert not (set(a) & set(b)), f"draws overlap: {sorted(set(a) & set(b))}"
print(f"\ndraws are disjoint; combined {len(a) + len(b)} of the correct items")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Checking the question text pins the
# row-to-image mapping every caption below depends on -- attaching a caption
# to the wrong page would make the whole audit worse than useless.
assert len(sample) == len(run), f"{len(sample)} items vs {len(run)} rows"
bad = [i for i in range(len(run))
       if sample[i]["orig_q"].strip() != str(run.iloc[i]["orig_q"]).strip()]
assert not bad, (
    f"{len(bad)} rows where the rebuilt sample's question does not match the "
    f"CSV's (first: {bad[:5]}). Images would be attached to the wrong rows.")
print("\nsample order matches the CSV on all 300 rows -- images are index-aligned")

# Every spot-check item must actually be one strict_v1 called correct.
scored = pilot.rescore.rescore_run(run, "strict_v1", progress=True)
correct_idx = set(scored.index[scored["transcription_correct"].astype(bool)])
for k, (df, _, _) in sheets.items():
    bad = sorted(set(df["item"]) - correct_idx)
    assert not bad, f"{k} holds items strict_v1 did NOT score correct: {bad}"
print(f"every sheet item is strict_v1-CORRECT (of {len(correct_idx)} such items)")

run  300 rows, model=['Qwen/Qwen2.5-VL-3B-Instruct']
  first40   40 items, 4 known false passes -> spotcheck_strict_v1_correct
  extra60   60 items, 0 known false passes -> spotcheck_strict_v1_correct_extra60

draws are disjoint; combined 100 of the correct items


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

...

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]


sample order matches the CSV on all 300 rows -- images are index-aligned


strict_v1:   0%|          | 0/300 [00:00<?, ?it/s]

every sheet item is strict_v1-CORRECT (of 141 such items)


In [3]:
# Contact sheets, same format as notebook 17 section 6. One Drive folder per draw.
import matplotlib.pyplot as plt

import pilot.corrected


# Tier names shortened: "display_math" alone eats a quarter of the caption
# width, and the header line overflowed into the neighbouring cell at full
# length -- caught by rendering a sheet and looking at it, not by the asserts.
TIER_ABBR = {"display_math": "disp", "last_line": "line", "option": "opt",
             "parse_fail": "FAIL", "inline_math": "inl"}
CAP_W = 52          # characters that fit one cell at ncols=3, fontsize 6


def caption_for(row):
    """Both automatic views of the item, so the coder can tell WHERE a false
    pass came from without leaving the sheet.

    `span` is what `extract_final_answer` picked off the page; `label` is what
    the comparison actually used after normalization. Showing them together
    separates the two ways a false pass is manufactured -- the extractor
    choosing the wrong or partial span, versus SymPy collapsing a good span to
    a single symbol. On the first 40 those ran 14 and 2 respectively, so the
    span line is usually the informative one.

    "strict_v1=CORRECT" is deliberately NOT repeated per cell -- it is true of
    every item on the sheet and the title already says so, and the space it
    freed is what stops the header colliding with the next cell.

    Human image reading overrides both.
    """
    i = int(row["item"])
    v = pilot.corrected.label_views(run, i)

    def z(x, n):
        s = " ".join(str(x).split())
        return s[:n] + ("..." if len(s) > n else "")

    marks = []
    if row["known_false_pass"]:
        marks.append("KNOWN-FALSE-PASS")
    if v["collapsed"]:
        marks.append("collapse?")
    head = f"item {i}   H={v['entropy']:.2f}"
    if marks:
        head += "   " + " ".join(marks)
    mt = TIER_ABBR.get(v["model_tier"], v["model_tier"])
    tt = TIER_ABBR.get(v["truth_tier"], v["truth_tier"])
    return "\n".join([
        z(head, CAP_W),
        z(f"span  M[{mt}]: {v['model_span']}", CAP_W),
        z(f"span  T[{tt}]: {v['truth_span']}", CAP_W),
        z(f"label M: {v['model_label']}", CAP_W),
        z(f"label T: {v['truth_label']}", CAP_W),
    ])


written = {}
for name, (df, subdir, stem) in sheets.items():
    sheet_dir = f"{PROJECT_DIR}/figures/{subdir}"
    os.makedirs(sheet_dir, exist_ok=True)
    items = df["item"].astype(int).tolist()
    figs = pilot.plotting.contact_sheet(
        [sample[i]["image"] for i in items],
        [caption_for(r) for _, r in df.iterrows()],
        ncols=3, per_page=9, cell_height=4.8, caption_fontsize=6.0,
        title=f"strict_v1 scored these CORRECT - read for FALSE PASSES ({name})")
    paths = []
    for page, fig in enumerate(figs, 1):
        path = f"{sheet_dir}/{stem}_p{page}.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor())
        plt.close(fig)
        paths.append(path)
    df.to_csv(f"{sheet_dir}/coding_sheet.csv", index=False)
    written[name] = paths
    print(f"{name}: {len(items)} items -> {len(figs)} page(s) in {subdir}/")
    for p in paths:
        print(f"    {os.path.basename(p)}")
    print(f"    coding_sheet.csv")

print("\nopen: My Drive > uncertainty-math-vlm > figures >")
for _, (_, subdir, _) in sheets.items():
    print(f"         {subdir}")
print("\nFill in final_label per item:")
print("  true_correct      - the model really did get it right")
print("  extraction_issue  - scored correct but the verdict was not earned (what we are hunting)")
print("  needs_visual      - cannot decide from the page")


first40: 40 items -> 5 page(s) in spotcheck_strict_v1_correct/
    spotcheck_correct_p1.png
    spotcheck_correct_p2.png
    spotcheck_correct_p3.png
    spotcheck_correct_p4.png
    spotcheck_correct_p5.png
    coding_sheet.csv
extra60: 60 items -> 7 page(s) in spotcheck_strict_v1_correct_extra60/
    spotcheck_correct_extra_p1.png
    spotcheck_correct_extra_p2.png
    spotcheck_correct_extra_p3.png
    spotcheck_correct_extra_p4.png
    spotcheck_correct_extra_p5.png
    spotcheck_correct_extra_p6.png
    spotcheck_correct_extra_p7.png
    coding_sheet.csv

open: My Drive > uncertainty-math-vlm > figures >
         spotcheck_strict_v1_correct
         spotcheck_strict_v1_correct_extra60

Fill in final_label per item:
  true_correct      - the model really did get it right
  extraction_issue  - scored correct but the verdict was not earned (what we are hunting)
  needs_visual      - cannot decide from the page


In [ ]:
# High-priority strict_v2 audit: export one PNG per item next to the sheet.
#
# The HTML references images/itemNNN.png RELATIVELY, so the images, the CSV
# and the HTML must end up in the same folder. Everything is written to one
# Drive directory you can then download whole and open offline.
import shutil

import pilot.strict_v2

HP_DIR = f"{PROJECT_DIR}/figures/strict_v2_high_priority"
HP_IMG = f"{HP_DIR}/images"
os.makedirs(HP_IMG, exist_ok=True)

queue = pd.read_csv("repo/reference/audit/strict_v2_review_queue_20260812.csv")
hp = pilot.strict_v2.high_priority_audit_sheet(
    queue,
    f"{HP_DIR}/strict_v2_high_priority_human_audit_20260812.csv",
    f"{HP_DIR}/strict_v2_high_priority_human_audit_20260812.html")

# Every item the sheet references must get an image, or the page shows a
# placeholder and the audit is done blind on that row.
missing = []
for i in hp["item_id"].astype(int):
    dest = f"{HP_IMG}/item{i:03d}.png"
    if not os.path.exists(dest):
        sample[i]["image"].save(dest)
    if not os.path.exists(dest):
        missing.append(i)
assert not missing, f"images failed to write for items: {missing}"

n_img = len([f for f in os.listdir(HP_IMG) if f.endswith(".png")])
assert n_img >= len(hp), f"{n_img} images for {len(hp)} rows"
print(f"{len(hp)} high-priority items -> {HP_DIR}")
print(f"   images/         {n_img} PNGs")
print(f"   ...audit_20260812.csv")
print(f"   ...audit_20260812.html   <- open this one")
print("\nDownload the whole strict_v2_high_priority folder, then open the HTML.")
print("Enter labels in the browser; 'Export CSV' downloads what you coded.")
print("Your reading of the image OVERRIDES every automatic label on the page.")
